# Notebook 03 – Predict 2025-26 NBA All-Stars

**What this notebook does:**
1. Loads the trained pipeline from `models/pipeline.joblib`.
2. Fetches 2025-26 player game logs from the NBA API (Oct 1, 2025 – Feb 1, 2026).
3. Builds features using the **same** preprocessing as training.
4. Runs `pipeline.predict_proba` and ranks players.
5. Outputs the top 24 predicted All-Stars with probabilities.

**Prerequisites**: run notebook 01 and notebook 02 first.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
TARGET_SEASON = "2025-26"   # prediction target
CUTOFF_DATE   = "2026-02-01"

DATA_DIR      = REPO_ROOT / "data"
CACHE_DIR     = DATA_DIR / "cache"
MODELS_DIR    = REPO_ROOT / "models"
PIPELINE_PATH = MODELS_DIR / "pipeline.joblib"
META_PATH     = MODELS_DIR / "pipeline_meta.json"

CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import joblib
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Load pipeline
pipeline = joblib.load(PIPELINE_PATH)
print("Pipeline loaded.")

# Load metadata (feature schema)
with open(META_PATH) as f:
    meta = json.load(f)

FEATURE_COLS = meta["feature_cols"]
print(f"Expected feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS[:5]} …")


## Build 2025-26 Features (Oct 1, 2025 → Feb 1, 2026)

In [ ]:
from src.feature_engineering import build_features_for_season

feats_2526 = build_features_for_season(
    season=TARGET_SEASON,
    cache_dir=CACHE_DIR,
)

print(f"Players with data: {len(feats_2526)}")
feats_2526.head(3)


In [ ]:
# Align feature columns to match training schema
# Any column in FEATURE_COLS that is missing gets filled with 0 / NaN
X_pred = feats_2526.reindex(columns=FEATURE_COLS, fill_value=0)

# Keep player metadata for the output table
player_meta = feats_2526[["PLAYER_ID", "PLAYER_NAME", "team", "GP"]].copy()
print(f"Feature matrix shape: {X_pred.shape}")


## Predict and Rank

In [ ]:
probas = pipeline.predict_proba(X_pred)[:, 1]
player_meta = player_meta.copy()
player_meta["ALLSTAR_PROB"] = probas

# Rank players by predicted probability
player_meta = player_meta.sort_values("ALLSTAR_PROB", ascending=False).reset_index(drop=True)
player_meta.index += 1   # 1-based rank

print(f"Top 24 predicted All-Stars for the {TARGET_SEASON} season")
print(f"(stats aggregated through {CUTOFF_DATE})\n")
top24 = player_meta.head(24)
print(top24[["PLAYER_NAME", "team", "GP", "ALLSTAR_PROB"]].to_string())


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top24["PLAYER_NAME"].iloc[::-1], top24["ALLSTAR_PROB"].iloc[::-1], color="steelblue")
ax.set_xlabel("Predicted All-Star Probability")
ax.set_title(f"Top 24 Predicted NBA All-Stars – {TARGET_SEASON} Season\n(features through {CUTOFF_DATE})")
ax.set_xlim(0, 1)
for i, (name, prob) in enumerate(zip(top24["PLAYER_NAME"].iloc[::-1], top24["ALLSTAR_PROB"].iloc[::-1])):
    ax.text(prob + 0.01, i, f"{prob:.1%}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(DATA_DIR / "top24_2025_26.png", dpi=150)
plt.show()
print("Plot saved to data/top24_2025_26.png")


In [ ]:
# Save full ranking to CSV
output_path = DATA_DIR / "predictions_2025_26.csv"
player_meta.to_csv(output_path)
print(f"Full ranking saved to {output_path}")
